In [8]:
from __future__ import annotations

import json
import sys
import types
from pathlib import Path
from typing import Any

import pandas as pd
import soccerdata as sd
from IPython.display import display
from pymongo import MongoClient, UpdateOne

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "dev_scripts" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from prod_pipeline.game_lineups import infer_formation
from prod_pipeline.helper import load_app_config, patch_soccerdata

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
OUTPUT_DIR = PROJECT_ROOT / "dev_scripts" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patch_soccerdata()
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
print(PROJECT_ROOT)

/Users/mario_omescu/Library/Mobile Documents/com~apple~CloudDocs/Sports Analytics/WhoScored Events Data


In [9]:
config = load_app_config(str(CONFIG_PATH))

# Edit these before running the repair cells.
SEASON_YEAR = "2019-2020"
SEASON_SHORT = "1920"
TARGET_GAME_IDS: list[int] = []
AUTO_DETECT_SUSPECT_GAMES = True
WRITE_CSV_OUTPUTS = True

LEAGUE = config.season.league
COMPETITION_NAME = config.season.name
COMPETITION_COUNTRY = config.season.country
FORMATION_MAPPING = config.formation_mapping
LINEUP_COLUMNS = config.game_lineups
FORMATION_COLUMNS = config.game_formations

mongo_cfg = config.mongo
collections = mongo_cfg.collections
client = MongoClient(mongo_cfg.url)
db = client[mongo_cfg.db]

collection_schedule = db[collections["collection_schedule"]]
collection_teams = db[collections["collection_teams"]]
collection_lineups = db[collections["collection_lineups"]]
collection_formations = db[collections["collection_formations"]]

In [10]:
schedule_docs = list(
    collection_schedule.find(
        {"season": SEASON_YEAR, "game_status": "finished"},
        {"_id": 0},
    ).sort("game_date", 1)
)
finished_games = pd.DataFrame(schedule_docs)
if finished_games.empty:
    raise ValueError(f"No finished games found for season={SEASON_YEAR}")

finished_games["game_id"] = finished_games["game_id"].astype(int)
available_teams = pd.DataFrame(
    collection_teams.find({}, {"_id": 0, "ws_team_id": 1, "ws_team_name": 1})
).rename(columns={"ws_team_id": "team_id", "ws_team_name": "team_name"})

ws = sd.WhoScored(leagues=LEAGUE, seasons=[SEASON_SHORT], headless=True)

def mongo_finished_schedule(ws_self, force_cache=False):
    return pd.DataFrame(
        {
            "league": LEAGUE,
            "season": SEASON_SHORT,
            "game": finished_games["game_id"].map(lambda x: f"match_{int(x)}"),
            "game_id": finished_games["game_id"].astype(int),
        }
    )

ws.read_schedule = types.MethodType(mongo_finished_schedule, ws)
display(finished_games[["game_id", "home_team_name", "away_team_name", "week"]].head())
print(f"Finished games: {finished_games['game_id'].nunique()}")

[07/26/26 16:10:59] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/WhoScored     ]8;id=7607984;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=7607985;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

[07/26/26 16:11:00] INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=7607990;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=7607991;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

,game_id,home_team_name,away_team_name,week
0,1388146,Bayern Munich,Hertha Berlin,1
1,1388139,Bayer Leverkusen,Paderborn,1
2,1388138,Borussia Dortmund,Augsburg,1
3,1388144,Freiburg,Mainz 05,1
4,1388143,Werder Bremen,Fortuna Duesseldorf,1


Finished games: 306


In [11]:
def normalize_mongo_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, dict):
        return {k: normalize_mongo_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, tuple):
        return [normalize_mongo_value(v) for v in value]
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def records_for_mongo(df: pd.DataFrame) -> list[dict]:
    return [
        {k: normalize_mongo_value(v) for k, v in record.items()}
        for record in df.to_dict(orient="records")
    ]


def extract_whoscored_players(filepath: str, game_id: int | None = None) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as file:
        match = json.load(file)

    if not isinstance(match, dict):
        raise ValueError(f"Invalid match data in {filepath}")

    if game_id is None:
        game_id = match.get("matchId")

    match_end = match.get("expandedMaxMinute") or match.get("maxMinute") or 90
    rows = []

    for field in ("home", "away"):
        team = match.get(field)
        if not isinstance(team, dict):
            continue

        team_id = team.get("teamId")
        red_cards = {}
        for event in team.get("incidentEvents") or []:
            if not isinstance(event, dict):
                continue
            card_type = event.get("cardType")
            if (
                event.get("playerId") is not None
                and isinstance(card_type, dict)
                and card_type.get("displayName") in {"Red", "SecondYellow"}
            ):
                red_cards[int(event["playerId"])] = event.get("expandedMinute", match_end)

        for player in team.get("players") or []:
            player_id = player.get("playerId")
            if player_id is None:
                continue

            player_id = int(player_id)
            is_starter = bool(player.get("isFirstEleven", False))
            minute_start = player.get("subbedInExpandedMinute", 0 if is_starter else 0)
            minute_end = red_cards.get(
                player_id,
                player.get("subbedOutExpandedMinute", match_end),
            )
            minutes_played = max(0, int(minute_end) - int(minute_start))

            rows.append(
                {
                    "game_id": game_id,
                    "team_id": int(team_id) if team_id is not None else None,
                    "player_id": player_id,
                    "player_name": player.get("name"),
                    "is_starter": is_starter,
                    "minutes_played": minutes_played,
                    "jersey_number": int(player["shirtNo"]) if player.get("shirtNo") not in (None, "") else None,
                    "starting_position": player.get("position"),
                }
            )

    return pd.DataFrame(
        rows,
        columns=[
            "game_id",
            "team_id",
            "player_id",
            "player_name",
            "is_starter",
            "minutes_played",
            "jersey_number",
            "starting_position",
        ],
    )


def process_game_lineups(df_players: pd.DataFrame, game_info: pd.DataFrame) -> pd.DataFrame:
    df = df_players.copy()
    df["_source_order"] = range(len(df))
    df = df.rename(columns={"is_starter": "starting_lineup"})

    game_id = int(game_info["game_id"].iloc[0])
    df["game_id"] = game_id
    df["season"] = SEASON_YEAR
    df["competition_name"] = COMPETITION_NAME
    df["competition_country"] = COMPETITION_COUNTRY

    for col in ["game_id", "team_id", "player_id", "minutes_played", "jersey_number"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    if "starting_lineup" in df.columns:
        df["starting_lineup"] = df["starting_lineup"].fillna(False).astype(bool)

    metadata_cols = [
        "game_id",
        "game_date",
        "week",
        "home_team_id",
        "home_team_name",
        "away_team_id",
        "away_team_name",
    ]
    metadata = game_info[[c for c in metadata_cols if c in game_info.columns]].drop_duplicates("game_id")
    df = df.merge(metadata, on="game_id", how="left")

    if "minutes_played" in df.columns:
        df["played"] = pd.to_numeric(df["minutes_played"], errors="coerce").fillna(0).gt(0)
    else:
        df["played"] = False

    if {"team_id", "home_team_id"}.issubset(df.columns):
        df["game_venue"] = df["team_id"].eq(df["home_team_id"]).map({True: "home", False: "away"})
    else:
        df["game_venue"] = None

    df = df.merge(available_teams, on="team_id", how="left")
    return df[LINEUP_COLUMNS + ["_source_order"]]


def build_formations(lineups: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    if lineups.empty:
        return pd.DataFrame(columns=FORMATION_COLUMNS), []

    starters = lineups.loc[lineups["starting_lineup"].fillna(False)].copy()
    if starters.empty:
        return pd.DataFrame(columns=FORMATION_COLUMNS), []

    formation_rows = []
    unknown_patterns = []

    for _, team_data in starters.sort_values("_source_order").groupby("team_id", sort=False):
        positions = [str(position) for position in team_data["starting_position"].tolist() if pd.notna(position)]
        if not positions:
            continue

        position_string = "-".join(positions)
        mapped_formation = FORMATION_MAPPING.get(position_string)
        if mapped_formation is None:
            mapped_formation = infer_formation(position_string)

        base_row = team_data.iloc[0]
        row = {
            "game_id": base_row.get("game_id"),
            "team_id": base_row.get("team_id"),
            "season": base_row.get("season"),
            "competition_name": base_row.get("competition_name"),
            "competition_country": base_row.get("competition_country"),
            "formation": mapped_formation,
            "game_date": base_row.get("game_date"),
            "game_venue": base_row.get("game_venue"),
            "positions": position_string,
            "team_name": base_row.get("team_name"),
            "week": base_row.get("week"),
            "opponent_team_name": None,
            "opponent_team_id": None,
            "opponent_formation": None,
            "opponent_positions": None,
            "is_new_pattern": position_string not in set(FORMATION_MAPPING),
        }
        formation_rows.append(row)

        if position_string not in FORMATION_MAPPING:
            unknown_patterns.append(
                {
                    "game_id": int(base_row["game_id"]),
                    "team_id": int(base_row["team_id"]),
                    "team_name": base_row.get("team_name"),
                    "positions": position_string,
                    "suggested_formation": mapped_formation,
                }
            )

    formations = pd.DataFrame(formation_rows)
    if formations.empty:
        return formations.reindex(columns=FORMATION_COLUMNS), unknown_patterns

    for _, row_indexes in formations.groupby("game_id").indices.items():
        if len(row_indexes) != 2:
            continue

        left_idx, right_idx = list(row_indexes)
        left = formations.loc[left_idx]
        right = formations.loc[right_idx]

        formations.at[left_idx, "opponent_team_name"] = right.get("team_name")
        formations.at[left_idx, "opponent_team_id"] = right.get("team_id")
        formations.at[left_idx, "opponent_formation"] = right.get("formation")
        formations.at[left_idx, "opponent_positions"] = right.get("positions")

        formations.at[right_idx, "opponent_team_name"] = left.get("team_name")
        formations.at[right_idx, "opponent_team_id"] = left.get("team_id")
        formations.at[right_idx, "opponent_formation"] = left.get("formation")
        formations.at[right_idx, "opponent_positions"] = left.get("positions")

    return formations[FORMATION_COLUMNS], unknown_patterns


def cached_event_path(game_id: int) -> Path:
    return Path.home() / "soccerdata" / "data" / "WhoScored" / "events" / f"{LEAGUE}_{SEASON_SHORT}" / f"{int(game_id)}.json"


def fetch_players_for_game(game_id: int) -> tuple[pd.DataFrame, str]:
    try:
        loader = ws.read_events(
            match_id=int(game_id),
            output_fmt="loader",
            retry_missing=True,
            on_error="skip",
        )
        players = loader.players(game_id=int(game_id))
        if players is not None and not players.empty:
            return players, "socceraction_loader"
    except Exception as exc:
        print(f"Falling back to cached JSON for game_id={game_id}: {type(exc).__name__}: {exc}")

    path = cached_event_path(game_id)
    if not path.exists():
        raise FileNotFoundError(f"No cached event file found for game_id={game_id}: {path}")

    return extract_whoscored_players(str(path), game_id=int(game_id)), "cached_json_fallback"

In [12]:
existing_lineups = pd.DataFrame(
    list(collection_lineups.find({"season": SEASON_YEAR}, {"_id": 0, "game_id": 1}))
)
existing_formations = pd.DataFrame(
    list(collection_formations.find({"season": SEASON_YEAR}, {"_id": 0, "game_id": 1}))
)

missing_games: list[int] = []
if AUTO_DETECT_SUSPECT_GAMES:
    finished_game_ids = set(finished_games["game_id"].astype(int).tolist())
    lineup_game_ids = set(existing_lineups["game_id"].astype(int).tolist()) if not existing_lineups.empty else set()
    formation_game_ids = set(existing_formations["game_id"].astype(int).tolist()) if not existing_formations.empty else set()
    missing_lineup_games = finished_game_ids - lineup_game_ids
    missing_formation_games = finished_game_ids - formation_game_ids
    missing_games = sorted(missing_lineup_games | missing_formation_games)

target_game_ids = sorted(set(TARGET_GAME_IDS or missing_games))
if not target_game_ids:
    raise ValueError("No target games selected. Set TARGET_GAME_IDS or enable AUTO_DETECT_SUSPECT_GAMES.")

print(f"Target games: {len(target_game_ids)}")
display(finished_games[finished_games['game_id'].isin(target_game_ids)][['game_id', 'home_team_name', 'away_team_name', 'week']].head(20))

Target games: 20


,game_id,home_team_name,away_team_name,week
284,1388370,Augsburg,Hoffenheim,32
286,1388349,Borussia Dortmund,Mainz 05,32
288,1388376,Bayern Munich,Freiburg,33
289,1388397,FC Koln,Eintracht Frankfurt,33
290,1388385,Fortuna Duesseldorf,Augsburg,33
291,1388388,Hertha Berlin,Bayer Leverkusen,33
292,1388382,Hoffenheim,Union Berlin,33
293,1388391,Mainz 05,Werder Bremen,33
294,1388400,Paderborn,Borussia M.Gladbach,33
295,1388379,RB Leipzig,Borussia Dortmund,33


In [13]:
repaired_lineups = []
repaired_formations = []
repair_summary = []
unmapped_formations = []
repair_errors = []

for game_id in target_game_ids:
    try:
        game_info = finished_games.loc[finished_games['game_id'].eq(int(game_id))].copy()
        if game_info.empty:
            raise ValueError(f"Game {game_id} not found in finished schedule")

        raw_players, source = fetch_players_for_game(int(game_id))
        lineups = process_game_lineups(raw_players, game_info)
        formations, unknown_patterns = build_formations(lineups)

        repaired_lineups.append(lineups[LINEUP_COLUMNS].copy())
        repaired_formations.append(formations.copy())
        unmapped_formations.extend(unknown_patterns)
        repair_summary.append(
            {
                "game_id": int(game_id),
                "source": source,
                "lineup_rows": int(len(lineups)),
                "formation_rows": int(len(formations)),
                "starter_rows": int(lineups['starting_lineup'].fillna(False).sum()),
            }
        )
    except Exception as exc:
        repair_errors.append({"game_id": int(game_id), "error": f"{type(exc).__name__}: {exc}"})

fixed_lineups = pd.concat(repaired_lineups, ignore_index=True) if repaired_lineups else pd.DataFrame(columns=LINEUP_COLUMNS)
fixed_formations = pd.concat(repaired_formations, ignore_index=True) if repaired_formations else pd.DataFrame(columns=FORMATION_COLUMNS)
repair_summary_df = pd.DataFrame(repair_summary)
repair_errors_df = pd.DataFrame(repair_errors)
unmapped_formations_df = pd.DataFrame(unmapped_formations)

if WRITE_CSV_OUTPUTS and not fixed_lineups.empty:
    fixed_lineups.to_csv(OUTPUT_DIR / f"workaround_{SEASON_SHORT}_lineups.csv", index=False)
if WRITE_CSV_OUTPUTS and not fixed_formations.empty:
    fixed_formations.to_csv(OUTPUT_DIR / f"workaround_{SEASON_SHORT}_formations.csv", index=False)

display(repair_summary_df)
display(repair_errors_df)
display(fixed_lineups.head())
display(fixed_formations.head())
display(unmapped_formations_df.head())

[07/26/26 16:11:05] INFO     [1/1] Retrieving game with id=1388349                                 ]8;id=7607996;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=7607997;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

Falling back to cached JSON for game_id=1388349: TypeError: 'NoneType' object is not subscriptable


                    INFO     [1/1] Retrieving game with id=1388370                                 ]8;id=7608002;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=7608003;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

Falling back to cached JSON for game_id=1388370: TypeError: 'NoneType' object is not subscriptable


                    INFO     [1/1] Retrieving game with id=1388376                                 ]8;id=7608008;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py\whoscored.py]8;;\:]8;id=7608009;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/whoscored.py#689\689]8;;\

...

,game_id,source,lineup_rows,formation_rows,starter_rows
0,1388349,cached_json_fallback,40,2,22
1,1388370,cached_json_fallback,40,2,22
2,1388376,cached_json_fallback,39,2,22
3,1388379,cached_json_fallback,40,2,22
4,1388382,cached_json_fallback,40,2,22
5,1388385,cached_json_fallback,40,2,22
6,1388388,cached_json_fallback,39,2,22
7,1388391,cached_json_fallback,40,2,22
8,1388394,cached_json_fallback,39,2,22
9,1388397,cached_json_fallback,40,2,22


""


,game_id,game_date,season,week,competition_name,competition_country,team_id,team_name,player_id,player_name,starting_lineup,played,minutes_played,jersey_number,starting_position,game_venue
0,1388349,2020-06-17 19:30:00,2019-2020,32,Bundesliga,Germany,44,Borussia Dortmund,63936,Roman Bürki,True,True,96,1,GK,home
1,1388349,2020-06-17 19:30:00,2019-2020,32,Bundesliga,Germany,44,Borussia Dortmund,111212,Emre Can,True,True,96,27,DC,home
2,1388349,2020-06-17 19:30:00,2019-2020,32,Bundesliga,Germany,44,Borussia Dortmund,21541,Mats Hummels,True,True,96,15,DC,home
3,1388349,2020-06-17 19:30:00,2019-2020,32,Bundesliga,Germany,44,Borussia Dortmund,11090,Lukasz Piszczek,True,True,55,26,DC,home
4,1388349,2020-06-17 19:30:00,2019-2020,32,Bundesliga,Germany,44,Borussia Dortmund,320834,Achraf Hakimi,True,True,96,5,DMR,home


,game_id,team_id,season,competition_name,competition_country,formation,game_date,game_venue,positions,team_name,week,opponent_team_name,opponent_team_id,opponent_formation,opponent_positions,is_new_pattern
0,1388349,44,2019-2020,Bundesliga,Germany,3-4-2-1,2020-06-17 19:30:00,home,GK-DC-DC-DC-DMR-DML-MC-MC-AMC-AMC-FW,Borussia Dortmund,32,Mainz 05,219,4-2-3-1,GK-DR-DC-DC-DL-DMC-DMC-AMR-AMC-AML-FW,False
1,1388349,219,2019-2020,Bundesliga,Germany,4-2-3-1,2020-06-17 19:30:00,away,GK-DR-DC-DC-DL-DMC-DMC-AMR-AMC-AML-FW,Mainz 05,32,Borussia Dortmund,44,3-4-2-1,GK-DC-DC-DC-DMR-DML-MC-MC-AMC-AMC-FW,False
2,1388370,1730,2019-2020,Bundesliga,Germany,4-2-3-1,2020-06-17 19:30:00,home,GK-DR-DC-DC-DL-DMC-DMC-AMR-AMC-AML-FW,Augsburg,32,Hoffenheim,1211,3-5-2,GK-DC-DC-DC-DMC-MR-MC-MC-ML-FW-FW,False
3,1388370,1211,2019-2020,Bundesliga,Germany,3-5-2,2020-06-17 19:30:00,away,GK-DC-DC-DC-DMC-MR-MC-MC-ML-FW-FW,Hoffenheim,32,Augsburg,1730,4-2-3-1,GK-DR-DC-DC-DL-DMC-DMC-AMR-AMC-AML-FW,False
4,1388376,37,2019-2020,Bundesliga,Germany,4-2-3-1,2020-06-20 14:30:00,home,GK-DR-DC-DC-DL-DMC-DMC-AMR-AMC-AML-FW,Bayern Munich,33,Freiburg,50,4-2-2-2,GK-DR-DC-DC-DL-DMC-DMC-AMC-AMC-FW-FW,False


""


In [14]:
DELETE_EXISTING_FOR_TARGET_GAMES = False

if fixed_lineups.empty and fixed_formations.empty:
    raise ValueError("Nothing to upload. Run the repair cell first.")

if DELETE_EXISTING_FOR_TARGET_GAMES:
    lineup_delete = collection_lineups.delete_many({"season": SEASON_YEAR, "game_id": {"$in": target_game_ids}})
    formation_delete = collection_formations.delete_many({"season": SEASON_YEAR, "game_id": {"$in": target_game_ids}})
    print(f"Deleted lineup docs: {lineup_delete.deleted_count}")
    print(f"Deleted formation docs: {formation_delete.deleted_count}")

lineup_ops = [
    UpdateOne(
        {
            "season": record["season"],
            "game_id": int(record["game_id"]),
            "team_id": int(record["team_id"]),
            "player_id": int(record["player_id"]),
        },
        {"$set": record},
        upsert=True,
    )
    for record in records_for_mongo(fixed_lineups)
    if record.get("team_id") is not None and record.get("player_id") is not None
]

formation_ops = [
    UpdateOne(
        {
            "season": record["season"],
            "game_id": int(record["game_id"]),
            "team_id": int(record["team_id"]),
        },
        {"$set": record},
        upsert=True,
    )
    for record in records_for_mongo(fixed_formations)
    if record.get("team_id") is not None
]

lineup_result = collection_lineups.bulk_write(lineup_ops, ordered=False) if lineup_ops else None
formation_result = collection_formations.bulk_write(formation_ops, ordered=False) if formation_ops else None

print(
    {
        "target_games": target_game_ids,
        "lineup_rows": len(fixed_lineups),
        "formation_rows": len(fixed_formations),
        "lineup_modified": 0 if lineup_result is None else lineup_result.modified_count + lineup_result.upserted_count,
        "formation_modified": 0 if formation_result is None else formation_result.modified_count + formation_result.upserted_count,
    }
)

{'target_games': [1388349, 1388370, 1388376, 1388379, 1388382, 1388385, 1388388, 1388391, 1388394, 1388397, 1388400, 1388403, 1388406, 1388409, 1388412, 1388415, 1388417, 1388418, 1388419, 1388420], 'lineup_rows': 796, 'formation_rows': 40, 'lineup_modified': 796, 'formation_modified': 40}
